In [10]:
from pathlib import Path
import sys
import importlib

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle


NB_DIR = Path.cwd().resolve()
PROJECT_DIR = NB_DIR.parent.parent
FLOW_DIR = PROJECT_DIR / "z.flow_postprocessing"
FLOW_SCRIPTS_DIR = FLOW_DIR / "scripts"

for path in [PROJECT_DIR, FLOW_SCRIPTS_DIR]:
    path = str(path)
    if path not in sys.path:
        sys.path.insert(0, path)

import theme.plot_theme as ptheme
import shcherbina_utils as shu

importlib.reload(ptheme)
ptheme.apply_theme()
importlib.reload(shu)
shu.apply_plot_style()

print(f"Notebook dir     : {NB_DIR}")
print(f"Project dir      : {PROJECT_DIR}")
print(f"Flow scripts dir : {FLOW_SCRIPTS_DIR}")



Notebook dir     : C:\Users\Jelle Gortemaker\Documents\Thesis\z.flow_postprocessing\notebooks
Project dir      : C:\Users\Jelle Gortemaker\Documents\Thesis
Flow scripts dir : C:\Users\Jelle Gortemaker\Documents\Thesis\z.flow_postprocessing\scripts


In [11]:
# NB_DIR = Path.cwd()

# case_name_uo = "JUL-1-2020-08-01_2020-08-16_ux"
# case_name_uv = "JUL-1-2020-08-01_2020-08-16_uy"
# case_name = "run_jul1_basilisk_2D"

# UO_FILE = (NB_DIR / f"../data/input/{case_name_uo}.nc").resolve()
# FILE = (NB_DIR / f"../data/input/{case_name}.nc").resolve()

# file = xr.open_dataset(UO_FILE)
# file

In [12]:

NB_DIR = Path.cwd()

case_name_uo = "FEB-1-2020-02-01_2020-02-15_ux"
case_name_uv = "FEB-1-2020-02-01_2020-02-15_uy"

UO_FILE = (NB_DIR / f"../data/input/{case_name_uo}.nc").resolve()
VO_FILE = (NB_DIR / f"../data/input/{case_name_uv}.nc").resolve()
OUTPUT = (NB_DIR / "run_feb1_basilisk_2D.nc").resolve()

# Names in the source files
UO_VAR = "uo"
VO_VAR = "vo"

# Names in the output file
UVEL_VAR = "UVEL"
VVEL_VAR = "VVEL"

LAT_MIN, LAT_MAX = 37.00, 40.00
LON_MIN, LON_MAX = -147.76, -144.24

EARTH_RADIUS_M = 6_371_000.0


def prepare(ds, variable):
    assert variable in ds.data_vars, (
        f"{variable!r} not found. "
        f"Available variables: {list(ds.data_vars)}"
    )
    assert "latitude" in ds.coords, "'latitude' coordinate not found"
    assert "longitude" in ds.coords, "'longitude' coordinate not found"

    ds = ds[[variable]]

    # Convert 0–360 longitude to −180–180 if necessary.
    if ds.longitude.max().item() > 180:
        ds = ds.assign_coords(
            longitude=((ds.longitude + 180) % 360) - 180
        )

    ds = ds.sortby("latitude").sortby("longitude")

    cropped = ds.sel(
        latitude=slice(LAT_MIN, LAT_MAX),
        longitude=slice(LON_MIN, LON_MAX),
    )

    assert cropped.sizes.get("latitude", 0) > 0, (
        "Empty latitude selection"
    )
    assert cropped.sizes.get("longitude", 0) > 0, (
        "Empty longitude selection"
    )

    return cropped


assert UO_FILE.is_file(), f"File not found: {UO_FILE}"
assert VO_FILE.is_file(), f"File not found: {VO_FILE}"

with xr.open_dataset(UO_FILE) as source_uo, \
     xr.open_dataset(VO_FILE) as source_vo:

    print("UO variables:", list(source_uo.data_vars))
    print("VO variables:", list(source_vo.data_vars))

    uo = prepare(source_uo, UO_VAR)
    vo = prepare(source_vo, VO_VAR)

    # Fail if time or horizontal coordinates differ.
    uo, vo = xr.align(uo, vo, join="exact")

    expected = xr.merge(
        [uo, vo],
        join="exact",
        compat="no_conflicts",
    )

    expected = expected.rename({
        UO_VAR: UVEL_VAR,
        VO_VAR: VVEL_VAR,
    })

    expected[UVEL_VAR].attrs["original_variable_name"] = UO_VAR
    expected[VVEL_VAR].attrs["original_variable_name"] = VO_VAR

    expected.attrs.update({
        "domain": "FEB-1 GLORYS box",
        "latitude_min": LAT_MIN,
        "latitude_max": LAT_MAX,
        "longitude_min": LON_MIN,
        "longitude_max": LON_MAX,
    })

    # Confirm the original variables are 2D in space.
    original_dims = ("time", "latitude", "longitude")

    assert expected[UVEL_VAR].dims == original_dims, (
        f"Unexpected UVEL dimensions: {expected[UVEL_VAR].dims}"
    )
    assert expected[VVEL_VAR].dims == original_dims, (
        f"Unexpected VVEL dimensions: {expected[VVEL_VAR].dims}"
    )

    # -------------------------------------------------------------
    # Add local Cartesian X/Y coordinates in metres.
    #
    # These coordinates are used by the existing notebook to infer
    # dx and dy. They do not alter the velocity data.
    # -------------------------------------------------------------
    latitude = expected.latitude.values.astype(np.float64)
    longitude = expected.longitude.values.astype(np.float64)

    reference_latitude_rad = np.deg2rad(np.mean(latitude))

    x_metres = (
        EARTH_RADIUS_M
        * np.cos(reference_latitude_rad)
        * np.deg2rad(longitude - longitude[0])
    )

    y_metres = (
        EARTH_RADIUS_M
        * np.deg2rad(latitude - latitude[0])
    )

    expected = expected.assign_coords(
        X=("longitude", x_metres),
        Y=("latitude", y_metres),
    )

    expected["X"].attrs.update({
        "long_name": "local eastward distance",
        "units": "m",
        "axis": "X",
        "comment": (
            "Approximate local Cartesian coordinate calculated from "
            "longitude using an equirectangular projection."
        ),
    })

    expected["Y"].attrs.update({
        "long_name": "local northward distance",
        "units": "m",
        "axis": "Y",
        "comment": (
            "Approximate local Cartesian coordinate calculated from "
            "latitude."
        ),
    })

    # Load all source data before closing the input files.
    expected.load()

    # Preserve the exact dataset before adding the synthetic Z layer.
    expected_2d = expected.copy(deep=True)

    # -------------------------------------------------------------
    # Add one synthetic vertical layer.
    #
    # Velocity values remain unchanged:
    # (time, latitude, longitude)
    # becomes
    # (time, Z, latitude, longitude)
    # -------------------------------------------------------------
    expected = expected.expand_dims(
        dim={"Z": np.array([0.0], dtype=np.float32)},
        axis=1,
    )

    expected["Z"].attrs.update({
        "long_name": "synthetic singleton vertical coordinate",
        "units": "m",
        "positive": "down",
        "axis": "Z",
        "comment": (
            "Added for compatibility with the 2D–3D postprocessing. "
            "The flow contains no resolved vertical structure."
        ),
    })

    expected.attrs["vertical_representation"] = (
        "Single synthetic layer at Z=0 m representing a 2D flow"
    )

    required_dims = ("time", "Z", "latitude", "longitude")

    assert expected[UVEL_VAR].dims == required_dims
    assert expected[VVEL_VAR].dims == required_dims
    assert expected.sizes["Z"] == 1


expected.to_netcdf(
    OUTPUT,
    engine="netcdf4",
)


# ---------------------------------------------------------------------
# Extensively verify the saved file
# ---------------------------------------------------------------------

with xr.open_dataset(OUTPUT) as result:
    result.load()

    # Verify the entire reopened dataset against what was written.
    xr.testing.assert_identical(result, expected)

    assert set(result.data_vars) == {UVEL_VAR, VVEL_VAR}
    assert UO_VAR not in result.data_vars
    assert VO_VAR not in result.data_vars

    # Verify the synthetic vertical dimension.
    assert "Z" in result.dims
    assert "Z" in result.coords
    assert result.sizes["Z"] == 1

    np.testing.assert_array_equal(
        result["Z"].values,
        np.array([0.0], dtype=np.float32),
    )

    required_dims = ("time", "Z", "latitude", "longitude")

    assert result[UVEL_VAR].dims == required_dims
    assert result[VVEL_VAR].dims == required_dims
    assert result[UVEL_VAR].shape == result[VVEL_VAR].shape

    # Verify that adding Z did not change any velocity values.
    xr.testing.assert_equal(
        result[UVEL_VAR].isel(Z=0, drop=True),
        expected_2d[UVEL_VAR],
    )

    xr.testing.assert_equal(
        result[VVEL_VAR].isel(Z=0, drop=True),
        expected_2d[VVEL_VAR],
    )

    lat = result.latitude.values
    lon = result.longitude.values
    x = result.X.values
    y = result.Y.values

    assert lat.size > 0
    assert lon.size > 0
    assert x.size == lon.size
    assert y.size == lat.size

    assert np.all(np.isfinite(lat))
    assert np.all(np.isfinite(lon))
    assert np.all(np.isfinite(x))
    assert np.all(np.isfinite(y))

    assert np.all(np.diff(lat) > 0), (
        "Latitude is not ascending and unique"
    )
    assert np.all(np.diff(lon) > 0), (
        "Longitude is not ascending and unique"
    )
    assert np.all(np.diff(x) > 0), (
        "X is not ascending and unique"
    )
    assert np.all(np.diff(y) > 0), (
        "Y is not ascending and unique"
    )

    assert LAT_MIN <= lat.min() <= lat.max() <= LAT_MAX
    assert LON_MIN <= lon.min() <= lon.max() <= LON_MAX

    time_index = result.indexes["time"]
    assert time_index.is_unique, "Duplicate timestamps found"
    assert time_index.is_monotonic_increasing, (
        "Time is not ordered"
    )

    dx = float(np.median(np.diff(x)))
    dy = float(np.median(np.diff(y)))

    assert np.isfinite(dx) and dx > 0
    assert np.isfinite(dy) and dy > 0

    for variable in (UVEL_VAR, VVEL_VAR):
        values = result[variable].values

        assert values.size > 0, f"{variable} is empty"
        assert np.isfinite(values).any(), (
            f"{variable} is entirely NaN"
        )

        print(
            f"{variable}: "
            f"dims={result[variable].dims}, "
            f"shape={result[variable].shape}, "
            f"dtype={result[variable].dtype}, "
            f"finite={np.isfinite(values).mean():.2%}"
        )

    print(result)
    print(f"Variables:          {list(result.data_vars)}")
    print(f"Z coordinate:       {result.Z.values}")
    print(f"Latitude range:     {lat.min()} to {lat.max()}")
    print(f"Longitude range:    {lon.min()} to {lon.max()}")
    print(f"Inferred dx:        {dx:.3f} m")
    print(f"Inferred dy:        {dy:.3f} m")
    print(f"Verified successfully: {OUTPUT}")

UO variables: ['uo']
VO variables: ['vo']
UVEL: dims=('time', 'Z', 'latitude', 'longitude'), shape=(121, 1, 514, 736), dtype=float64, finite=100.00%
VVEL: dims=('time', 'Z', 'latitude', 'longitude'), shape=(121, 1, 514, 736), dtype=float64, finite=100.00%
<xarray.Dataset> Size: 732MB
Dimensions:    (time: 121, Z: 1, latitude: 514, longitude: 736)
Coordinates:
  * time       (time) datetime64[ns] 968B 2020-01-31 ... 2020-02-14T23:59:59....
  * Z          (Z) float32 4B 0.0
  * latitude   (latitude) float64 4kB 37.0 37.0 37.01 37.01 ... 39.0 39.01 39.01
    Y          (latitude) float64 4kB 0.0 435.9 871.7 ... 2.232e+05 2.236e+05
  * longitude  (longitude) float64 6kB -147.8 -147.8 -147.7 ... -144.2 -144.2
    X          (longitude) float64 6kB 0.0 419.3 838.5 ... 3.077e+05 3.082e+05
Data variables:
    UVEL       (time, Z, latitude, longitude) float64 366MB 0.1895 ... 0.03448
    VVEL       (time, Z, latitude, longitude) float64 366MB -0.01031 ... -0.1206
Attributes:
    domain:        